<a href="https://colab.research.google.com/github/YoungSong99/demonhacks_26/blob/main/FastAPI_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastapi uvicorn nest_asyncio pyngrok

In [ ]:
%%writefile app.py
import os, uuid
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse, Response, JSONResponse

app = FastAPI()

BASE_DIR = "/content/eosn_api"
AGING_DIR = os.path.join(BASE_DIR, "aging")
MODEL_DIR = os.path.join(BASE_DIR, "models")
os.makedirs(AGING_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

@app.get("/")
def root():
    return {"ok": True}

@app.post("/api/age")
async def api_age(file: UploadFile = File(...), age: int = Form(...)):
    image_id = str(uuid.uuid4())
    ext = os.path.splitext(file.filename or "")[1].lower() or ".jpg"
    save_path = os.path.join(AGING_DIR, f"{image_id}{ext}")

    data = await file.read()
    with open(save_path, "wb") as f:
        f.write(data)

    return JSONResponse({
        "image_id": image_id,
        "received_age": age,
        "bytes": len(data),
    })

@app.get("/api/images/{image_id}")
def api_get_image(image_id: str):
    candidates = [fn for fn in os.listdir(AGING_DIR) if fn.startswith(image_id)]
    if not candidates:
        return Response(status_code=404)

    path = os.path.join(AGING_DIR, candidates[0])
    media = "image/png" if path.lower().endswith(".png") else "image/jpeg"
    return FileResponse(path, media_type=media)


@app.post("/api/build-3d")
async def api_build_3d(file: UploadFile = File(...)):
    model_id = str(uuid.uuid4())

    data = await file.read()
    # 입력 저장(디버그)
    with open(os.path.join(MODEL_DIR, f"{model_id}_input.jpg"), "wb") as f:
        f.write(data)

    model_path = os.path.join(MODEL_DIR, f"{model_id}.glb")
    with open(model_path, "wb") as f:
        f.write(b"GLB_DUMMY_MODEL_DATA")

    return JSONResponse({"model_id": model_id})

@app.get("/api/models/{model_id}")
def api_get_model(model_id: str):
    path = os.path.join(MODEL_DIR, f"{model_id}.glb")
    if not os.path.exists(path):
        return Response(status_code=404)
    return FileResponse(path, media_type="model/gltf-binary", filename=f"{model_id}.glb")

Overwriting app.py


In [ ]:
!ngrok config add-authtoken 23vTykUOXa4caAa7FyoVde6lLPD_7GGbfM75sLqoYhBFgQ6AM


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok
public_url = ngrok.connect(8000)
print(public_url)

NgrokTunnel: "https://a9a4-146-148-108-227.ngrok-free.app" -> "http://localhost:8000"


In [ ]:
!uvicorn app:app --host 127.0.0.1 --port 8000 --app-dir /content

INFO:     Started server process [32150]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     75.102.241.224:0 - "POST /api/age HTTP/1.1" 200 OK
INFO:     75.102.241.224:0 - "GET /api/images/c83a054e-2d2c-4424-9049-a79b00d67b13 HTTP/1.1" 200 OK
INFO:     75.102.241.224:0 - "GET /api/images/c83a054e-2d2c-4424-9049-a79b00d67b13 HTTP/1.1" 200 OK
INFO:     75.102.241.224:0 - "POST /api/build-3d HTTP/1.1" 200 OK
INFO:     75.102.241.224:0 - "GET /api/images/c83a054e-2d2c-4424-9049-a79b00d67b13 HTTP/1.1" 200 OK
